### PRUEBA TÉCNICA - RUEDATA
#### Cargo: Senior BI Analyst 
 Angelly Ortega - Estadística - Universidad del Valle, Cali, CO.

#### Librerías

In [0]:
import re

from pyspark.sql import functions as F

from pyspark.sql.functions import (
    col, count, when, lit, avg, stddev, min, max, mean, 
    date_sub, datediff, trim, upper, regexp_replace, round
)
from pyspark.sql.types import DoubleType, IntegerType



#### Cargue de datos - Load Data
Se realiza el cargue de la información con la cual se va a trabajar.
The information to be worked with is loaded.

In [0]:
def load_data_spark(spark, table_name, server, database, username, password, schema=None):
    """
    Load a table from SQL Server into a Spark DataFrame.
    - 'spark' must be an active SparkSession (passed in).
    """
    # Manejo de nombres con espacios
    table_name_formatted = f"[{table_name}]"
    schema_formatted = f"[{schema}]" if schema else None
    
    if schema_formatted:
        full_table_name = f"{schema_formatted}.{table_name_formatted}"
    else:
        full_table_name = table_name_formatted
    
    # URL JDBC 
    jdbc_url = f"jdbc:sqlserver://{server};databaseName={database};encrypt=true;trustServerCertificate=true;"
    
    # Leer usando el driver explícito
    df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", full_table_name) \
        .option("user", username) \
        .option("password", password) \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()
    
    return df

def load_spaceparts(spark, server, database, username, password):
    tables = {
        # DimView
        #'brands': ('DimView', 'Brands'),
        #'budget_rate': ('DimView', 'Budget Rate'),
        'customers': ('DimView', 'Customers'),
        #'employees': ('DimView', 'Employees'),
        #'exchange_rate': ('DimView', 'Exchange Rate'),
        #'invoice_doc_type': ('DimView', 'Invoice Document Type'),
        #'order_doc_type': ('DimView', 'Order Document Type'),
        #'order_status': ('DimView', 'Order Status'),
        'products': ('DimView', 'Products'),
        'regions': ('DimView', 'Regions'),
        # FactView
        #'budget': ('FactView', 'Budget'),
        #'forecast': ('FactView', 'Forecast'),
        'invoices': ('FactView', 'Invoices'),
        'orders': ('FactView', 'Orders')
    }
    
    result = {}
    for key, (schema, table) in tables.items():
        try:
            df = load_data_spark(spark, table, server, database, username, password, schema)
            result[key] = df
            print(f"Successfully loaded {schema}.{table} (key={key})")
        except Exception as e:
            print(f"Error loading {schema}.{table}: {e}")
    return result


In [0]:
# parámetros y ejecución
server = "te3-training-eu.database.windows.net"
database = "SpacePartsCoDW"
username = "dwreader"
password = "TE3#reader!"

all_tables = load_spaceparts(spark, server, database, username, password)


Cantidad de datos en cada tabla. Amount of data in each table.

In [0]:
all_tables['orders'].count()

In [0]:
all_tables['invoices'].count()

Se extrae cada tabla y para las tablas de hechos, debido a la cantidad de datos se extrae solo el primer millon de datos.

In [0]:
products_df = all_tables['products']
print(products_df.count())
products_df.display()

In [0]:
customers_df = all_tables['customers']
print(customers_df.count())
customers_df.display()

In [0]:
regions_df = all_tables['regions']
print(regions_df.count())
regions_df.display()

In [0]:
invoices_df = all_tables['invoices'].limit(10**6).cache()
print(invoices_df.count())
invoices_df.display()

In [0]:
orders_df = all_tables['orders'].limit(10**6).cache()
print(orders_df.count())
orders_df.display()

In [0]:

print([(f.name, f.dataType.simpleString()) for f in invoices_df.schema.fields])

### Limpieza y Transformación

In [0]:
#--------------------------
# Cambiar formato a string
#--------------------------

# Lista de columnas para pasar a string
cols_to_cast = ["Billing Document Number", "Billing Document Line Item Number", "Product Key"]

# Aplicar cast a todas las columnas
for c in cols_to_cast:
    invoices_df = invoices_df.withColumn(c, col(c).cast("string"))


# Verificar los nuevos tipos de datos
invoices_df.printSchema()

In [0]:
#--------------------------
# Cambiar formato a string
#--------------------------

# Lista de columnas para pasar a string
cols_to_cast = ["Sales Order Document Number", "Sales Order Document Line Item Number", "Product Key"]

# Aplicar cast a todas las columnas
for c in cols_to_cast:
    orders_df = orders_df.withColumn(c, col(c).cast("string"))


# Verificar los nuevos tipos de datos
orders_df.printSchema()

In [0]:
#--------------------------
# Cambiar formato a string
#--------------------------

# Lista de columnas para pasar a string
cols_to_cast = ["Product Key"]

# Aplicar cast a todas las columnas
for c in cols_to_cast:
    products_df = products_df.withColumn(c, col(c).cast("string"))


# Verificar los nuevos tipos de datos
products_df.printSchema()

In [0]:
def null_percentage_report(df, df_name="DataFrame"):
    total_rows = df.count()
    
    # Conteo de nulos y porcentaje
    null_report = df.select([
        count(when(col(c).isNull(), c)).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    
    print(f"\n>>> Reporte de nulos para {df_name} (total filas = {total_rows})")
    
    for col_name, null_count in null_report.items():
        null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
        print(f"{col_name:30} -> {null_count} nulos ({null_pct:.2f}%)")


In [0]:
null_percentage_report(regions_df, "regions_df")

In [0]:
null_percentage_report(products_df, "products_df")

In [0]:
null_percentage_report(customers_df, "customers_df")

In [0]:
null_percentage_report(orders_df, "orders_df")

In [0]:
null_percentage_report(invoices_df, "Invoices_df")

Para la BD Invoices, se tiene valores duplicados para Billing Document Number, esto se debe a que por factura se tiene Billing Document Line Item Number. Si se requiere conocer la cantidad total de facturas, se debe realizar con un distinctcount.

In [0]:
# Agrupar por la columna y contar ocurrencias
duplicates = (
    invoices_df.groupBy("Billing Document Number")
    .count()
    .filter(F.col("count") > 1)
)

duplicates.show()

In [0]:
# Filtrar por un número de documento específico
valor = "1000104935"

df_filtrado = invoices_df.filter(col("Billing Document Number") == valor)

df_filtrado.display()


#### Existen filas duplicadas?

- Orders

In [0]:
# Contar registros por todas las columnas
duplicates_orders = orders_df.groupBy(orders_df.columns) \
               .count() \
               .filter(F.col("count") > 1)

duplicates_orders.display()


- Invoices

In [0]:
# Identify duplicate rows in invoices_df based on specific columns
duplicate_rows_invoices = (
    invoices_df.groupBy("Billing Document Number", "Billing Document Line Item Number", "Product Key")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_rows_invoices)

In [0]:
# Contar ocurrencias de cada fila (todas las columnas)
df_duplicates = (
    invoices_df.groupBy(invoices_df.columns)  # agrupamos por todas las columnas del DataFrame
      .count()
      .filter(F.col("count") > 1)  # solo las filas que se repiten
)

df_duplicates.display()


In [0]:
# Contar registros por todas las columnas
duplicates_invoices = invoices_df.groupBy('Billing Document Number') \
                        .count() \
                        .filter(F.col("count") > 1)

duplicates_invoices.display()


In [0]:
# Agrupar por la columna y contar ocurrencias
duplicates = (
    orders_df.groupBy("Sales Order Document Number")
    .count()
    .filter(F.col("count") > 1)
)

duplicates.show()

In [0]:
# Ejemplo: filtrar por un número de documento específico
valor = "11437544"

df_filtrado = orders_df.filter(col("Sales Order Document Number") == valor)

df_filtrado.display()


In [0]:
# Eliminar duplicados
invoices_df = invoices_df.dropDuplicates()
orders_df = orders_df.dropDuplicates()
customers_df = customers_df.dropDuplicates()
products_df = products_df.dropDuplicates()
regions_df = regions_df.dropDuplicates()

#### Existe la llave en la tabla de dimensión?

In [0]:
def orphan_keys(fact_df, dim_df, fact_key, dim_key, name):
    orphans = fact_df.join(dim_df, fact_df[fact_key] == dim_df[dim_key], "left_anti").count()
    return [(name, fact_key, orphans)]

In [0]:
invoice_orphans_prod = orphan_keys(invoices_df, products_df, "Product Key", "Product Key", "Invoice")
invoice_orphans_prod

In [0]:
invoice_orphans_cust = orphan_keys(invoices_df, customers_df, "Customer Key", "Customer Key", "customers")
invoice_orphans_cust

In [0]:
orders_orphans_prod = orphan_keys(orders_df, products_df, "Product Key", "Product Key", "orders")
orders_orphans_prod

In [0]:
orders_orphans_cust = orphan_keys(orders_df, customers_df, "Customer Key", "Customer Key", "customers")
orders_orphans_cust

#### Existen outliers?

In [0]:
def outlier_report(df, numeric_cols, name):
    stats = []
    for c in numeric_cols:
        s = df.select(
            mean(col(c)).alias("mean"),
            stddev(col(c)).alias("std"),
            min(col(c)).alias("min"),
            max(col(c)).alias("max")
        ).collect()[0]
        stats.append((name, c, s["mean"], s["std"], s["min"], s["max"]))
    return spark.createDataFrame(stats, ["tabla", "columna", "mean", "std", "min", "max"])

In [0]:
numeric_cols = [field.name for field in orders_df.schema.fields if isinstance(field.dataType, (DoubleType, IntegerType))]
numeric_cols

### Conclusiones generales: Invoices
- Hay mucha dispersión y outliers en casi todas las métricas.
- Existen valores negativos en costos, penalidades, cantidades y facturas; estos pueden ser devoluciones, descuentos o errores de registro.
- La media en casi todas las variables está influenciada por valores extremos.
- Algunas columnas (como Taxes & Commercial Fees) no aportan información porque siempre son 0.

In [0]:
def basic_statistics(df, numeric_cols):
    stats = []
    for c in numeric_cols:
        s = df.select(
            mean(col(c).cast(DoubleType())).alias("mean"),
            stddev(col(c).cast(DoubleType())).alias("std"),
            min(col(c).cast(DoubleType())).alias("min"),
            max(col(c).cast(DoubleType())).alias("max")
        ).collect()[0]
        stats.append((c, s["mean"], s["std"], s["min"], s["max"]))
    return spark.createDataFrame(stats, ["columna", "mean", "std", "min", "max"])

numeric_cols = [field.name for field in invoices_df.schema.fields if isinstance(field.dataType, (DoubleType, IntegerType))]
invoices_stats = basic_statistics(invoices_df, numeric_cols)
display(invoices_stats)

### Conclusiones generales: Orders
Las cantidades pedidas son bastante consistentes y homogéneas; la variabilidad entre pedidos en cantidad es mínima, a diferencia de lo que pasa con el valor económico.

- En valor económico, las órdenes son muy variables (hay desde órdenes muy pequeñas hasta órdenes muy grandes).
- En cantidad de productos, las órdenes son bastante uniformes (entre 2 y 4 unidades), lo cual sugiere que la diferencia en valor proviene más de los precios de los productos que de la cantidad pedida.


In [0]:
def basic_statistics(df, numeric_cols):
    stats = []
    for c in numeric_cols:
        s = df.select(
            mean(col(c).cast(DoubleType())).alias("mean"),
            stddev(col(c).cast(DoubleType())).alias("std"),
            min(col(c).cast(DoubleType())).alias("min"),
            max(col(c).cast(DoubleType())).alias("max")
        ).collect()[0]
        stats.append((c, s["mean"], s["std"], s["min"], s["max"]))
    return spark.createDataFrame(stats, ["columna", "mean", "std", "min", "max"])

numeric_cols = [field.name for field in orders_df.schema.fields if isinstance(field.dataType, (DoubleType, IntegerType))]
orders_stats = basic_statistics(orders_df, numeric_cols)
display(orders_stats)

#### Transformaciones

In [0]:
invoices_df = invoices_df.withColumn(
    "Deducciones",
    F.col("Late Delivery Penalties") +
    F.col("Overdue Payment Penalties") +
    F.col("Taxes & Commercial Fees") +
    F.col("Freight") +
    F.col("Delivery Cost")
).withColumn(
    "Ganancia_Neta",
    F.col("Net Invoice Value") - (F.col("Net Invoice COGS") + F.col("Deducciones"))
)

In [0]:
# Columna 'clasific_value' y 'Net Invoice Value_adj'
invoices_df = invoices_df.withColumn(
    "clasific_value",
    F.when(F.col("Net Invoice Value") < 0, F.lit("Negativo"))
     .when(F.col("Net Invoice Value") > 0, F.lit("Positivo"))
     .otherwise(F.lit("Cero"))  # por si existieran valores = 0
).withColumn(
    "Net Invoice Value_abs",
    F.abs(F.col("Net Invoice Value"))  # convierte los negativos en positivos
)

# Ver resultados
invoices_df.select("Net Invoice Value", "clasific_value", "Net Invoice Value_abs").show(10)


In [0]:
# Columna Deductions_value
invoices_df = invoices_df.withColumn(
    "Deductions_value",
    F.when(F.col("clasific_value") == "Negativo", F.col("Net Invoice Value_abs"))
     .otherwise(F.lit(0))
)

# Ver resultado
invoices_df.select("Net Invoice Value", "Net Invoice Value_abs", "clasific_value", "Deductions_value").show(10)


In [0]:
# Columna 'clasific_quantity' y 'Net Invoice Quantity_adj'
invoices_df = invoices_df.withColumn(
    "clasific_quantity",
    F.when(F.col("Net Invoice Quantity") < 0, F.lit("Negativo"))
     .when(F.col("Net Invoice Quantity") > 0, F.lit("Positivo"))
     .otherwise(F.lit("Cero"))  # por si existieran valores = 0
).withColumn(
    "Net Invoice Quantity_abs",
    F.abs(F.col("Net Invoice Quantity"))  # convierte los negativos en positivos
)

# Ver resultado
invoices_df.select("Net Invoice Quantity", "clasific_quantity", "Net Invoice Quantity_abs").show(10)

In [0]:
# Columna Deductions_quantity
invoices_df = invoices_df.withColumn(
    "Deductions_quantity",
    F.when(F.col("clasific_quantity") == "Negativo", F.col("Net Invoice Quantity_abs"))
     .otherwise(F.lit(0))
)

# Ver resultado
invoices_df.select("Net Invoice Quantity", "Net Invoice Quantity_abs", "clasific_quantity", "Deductions_quantity").show(10)


#### Agrupación de Datos - Data Grouping

In [0]:
invoices_grouped = (
    invoices_df.groupBy(
        "Billing Date",      # Fecha de facturación
        #"Customer Key",      # Cliente
        "Product Key"        # Producto
    )
    .agg(
        F.sum("Delivery Cost").alias("Total_Delivery_Cost"),
        F.sum("Net Invoice COGS").alias("Total_COGS"),
        F.sum("Late Delivery Penalties").alias("Total_Late_Delivery_Penalties"),
        F.sum("Overdue Payment Penalties").alias("Total_Overdue_Payment_Penalties"),
        F.sum("Taxes & Commercial Fees").alias("Total_Taxes_Fees"),
        F.sum("Freight").alias("Total_Freight"),
        F.sum("Net Invoice Cost").alias("Total_Invoice_Cost"),
        F.sum("Net Invoice Value_abs").alias("Total_Invoice_Value"),
        F.sum("Deductions_value").alias("Total_deductions_value"),
        F.sum("Net Invoice Quantity_abs").alias("Total_Invoice_Qty"),
        F.sum("Deductions_quantity").alias("Total_deductions_quantity"),
        F.sum("Deducciones").alias("Total_deductions_add"),
        F.sum("Ganancia_Neta").alias("Total_Ganancia_Neta"),
        F.countDistinct("Billing Document Number").alias("Num_Invoices")
    )
)

In [0]:
invoices_df.count()

In [0]:
invoices_grouped.count()

In [0]:
orders_grouped = (
    orders_df.groupBy(
        "Order Date",        # Fecha de la orden
        #"Customer Key",      # Cliente
        "Product Key"        # Producto
    )
    .agg(
        F.sum("Net Order Value").alias("Total_Order_Value"),
        F.sum("Net Order Quantity").alias("Total_Order_Qty"),
        F.countDistinct("Sales Order Document Number").alias("Num_Orders")
    )
)

orders_grouped.count()

In [0]:
# ----------------------------
#  Limpieza de columnas
# ----------------------------
def clean_columns(df):
    # Normalizar nombres: minúsculas + reemplazar espacios por "_"
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower().replace(" ", "_"))
    return df

invoices_agg = clean_columns(invoices_grouped)
orders_agg = clean_columns(orders_grouped)
regions_df = clean_columns(regions_df)
customers_df = clean_columns(customers_df)
products_df = clean_columns(products_df)

In [0]:
products_df.display()

In [0]:
#invoices_agg.write.format("parquet").mode("overwrite").save("/mnt/data/invoices_agg")

In [0]:
#invoices_agg.write.format("parquet").mode("overwrite").save("/mnt/data/invoices_agg")
#orders_agg.write.format("parquet").mode("overwrite").save("/mnt/data/orders_agg")
#regions_df.write.format("parquet").mode("overwrite").save("/mnt/data/regions_df")
#customers_df.write.format("parquet").mode("overwrite").save("/mnt/data/customers_df")
#products_df.write.format("parquet").mode("overwrite").save("/mnt/data/products_df")

In [0]:
print([(f.name, f.dataType.simpleString()) for f in orders_agg.schema.fields])

In [0]:
print([(f.name, f.dataType.simpleString()) for f in invoices_agg.schema.fields])